In [131]:
! uv pip install langchain langchain-google-genai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.14.3 environment at: /Users/yashgiradkar/Development/projects/LLMops-Project/.venv
Resolved 73 packages in 1.17s                                        
Prepared 9 packages in 1.27s                                                 cryptography           ------------------------------ 3.70 MiB/3.85 MiB         cryptography           ------------------------------ 910.96 KiB/3.85 MiB       cryptography           ------------------------------ 238.96 KiB/3.85 MiB       cryptography           ------------------------------ 174.96 KiB/3.85 MiB       cryptography           ------------------------------ 158.96 KiB/3.85 MiB       cryptography           ------------------------------ 62.96 KiB/3.85 MiB        cryptography           ------------------------------ 32.00 KiB/3.85 MiB        cryptography           ------------------------------ 16.00 KiB/3.85 MiB        cryptography           ------------------------------     0 B/3.85 MiB          
Uninstalled 1 package in 6ms
Insta

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")

AQ.Ab8RN6I


Data Ingestion

In [118]:
!pip install -U langchain-community

In [119]:
from langchain_community.document_loaders import TextLoader

In [120]:
loader = TextLoader("/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt", encoding="utf8")
documents = loader.load()

In [121]:
documents[0].page_content[:500]  # Print the first 500 characters of the first document

'# MMR (Maximal Marginal Relevance) Implementation Guide\n\n## What is MMR?\n\n**Maximal Marginal Relevance (MMR)** is a retrieval algorithm that balances relevance and diversity in search results. Instead of just returning the most similar documents, MMR ensures variety by reducing redundancy.\n\n### How MMR Works:\n\n1. **Initial Retrieval**: Fetch `fetch_k` documents using similarity search\n2. **Re-ranking**: Select `k` documents that maximize:\n   - **Relevance** to the query\n   - **Diversity** from a'

In [122]:
!pip install -U langchain-text-splitters

In [123]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [124]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [125]:
text_chunks=text_splitter.split_documents(documents)

In [126]:
text_chunks

[Document(metadata={'source': '/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt'}, page_content='# MMR (Maximal Marginal Relevance) Implementation Guide\n\n## What is MMR?'),
 Document(metadata={'source': '/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt'}, page_content='**Maximal Marginal Relevance (MMR)** is a retrieval algorithm that balances relevance and diversity in search results. Instead of just returning the most similar documents, MMR ensures variety by'),
 Document(metadata={'source': '/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt'}, page_content='ensures variety by reducing redundancy.'),
 Document(metadata={'source': '/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt'}, page_content='### How MMR Works:'),
 Document(metadata={'source': '/Users/yashgiradkar/Development/projects/LLMops-Project/data/MMR_Guide.txt'}, page_content='1. **Initial Retrieval**: Fetch `fetch_k` d

In [127]:
! uv pip install faiss-cpu
! uv pip install -U langchain-openai langchain-community faiss-cpu


Using Python 3.14.3 environment at: /Users/yashgiradkar/Development/projects/LLMops-Project/.venv
Checked 1 package in 2ms
Using Python 3.14.3 environment at: /Users/yashgiradkar/Development/projects/LLMops-Project/.venv
Resolved 53 packages in 407ms                                        
Prepared 1 package in 2ms                                                
Uninstalled 1 package in 4ms
Installed 1 package in 4ms                                  
 - websockets==15.0.1
 + websockets==16.1.1


In [136]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [137]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [138]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [139]:
vectorstore

In [140]:
retriever=vectorstore.as_retriever()

In [141]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
### Key Parameters:
--------------------------------------------------
Document 2:
- For most conversational AI applications, the diversity benefits outweigh the minimal performance cost
--------------------------------------------------
Document 3:
ensures variety by reducing redundancy.
--------------------------------------------------
Document 4:
**Similarity Search** might return:
1. Attention mechanism definition
2. Another attention mechanism definition (redundant)
3. Attention mechanism benefits (similar)
--------------------------------------------------


In [144]:
from langchain_core.prompts import ChatPromptTemplate
template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [145]:
prompt=ChatPromptTemplate.from_template(template)

In [146]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [147]:
from langchain_core.output_parsers import StrOutputParser

In [148]:
output_parser=StrOutputParser()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

In [158]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [160]:
rag_chain.invoke("tell me about MRR")

'Maximal Marginal Relevance (MMR) is a retrieval algorithm designed to balance the relevance and diversity of search results. Unlike traditional methods that only return the most similar documents, MMR intentionally incorporates variety into the output. By doing so, it provides a more comprehensive understanding of the information being retrieved.'